# 异步 IO

## 协程 Coroutine
> 线程——执行一个子程序

> 协程——在子程序执行过程中可以中断，去执行其他子程序
>
> 在一个线程中执行，协程的切换没有线程切换的开销
>
> 不需要锁机制
>
> 这个例子里面可以看到生产者消费者是同步执行的，并且没有经过锁机制

- 补充生成器知识点
- `n = yield r`
> 返回 `r` 作为 `send()` 的结果
>
> 接收 `send()` 的参数作为 `n`
- `c.send(value)`
> 启动生成器
>
> 将 `value` 作为 `yield` 表达式的值
>
> 执行到下一次碰到 `yield`

In [1]:
# 这是一个Generator(包含了 yield 函数)
from _collections_abc import Iterable
def consumer():
    r = ''
    while True:
        n = yield r
        if not n:
            return
        print('[CONSUMER] Consuming %s...' % n)
        r = '200 OK'
print(isinstance(consumer(), Iterable))
def produce(c):
    c.send(None)
    n = 0
    while n < 5:
        n = n + 1
        print('[PRODUCER] Producing %s...' % n)
        r = c.send(n)
        print('[PRODUCER] Consumer return: %s' % r)
    c.close()

c = consumer()# 返回生成器对象
produce(c)


True
[PRODUCER] Producing 1...
[CONSUMER] Consuming 1...
[PRODUCER] Consumer return: 200 OK
[PRODUCER] Producing 2...
[CONSUMER] Consuming 2...
[PRODUCER] Consumer return: 200 OK
[PRODUCER] Producing 3...
[CONSUMER] Consuming 3...
[PRODUCER] Consumer return: 200 OK
[PRODUCER] Producing 4...
[CONSUMER] Consuming 4...
[PRODUCER] Consumer return: 200 OK
[PRODUCER] Producing 5...
[CONSUMER] Consuming 5...
[PRODUCER] Consumer return: 200 OK


## 使用 `asyncio`
> `@asyncio.coroutine` 可以把 `generator` 标记成 `coroutine`
>
> `coroutine` 内部用 `yield from` 调用另一个 `coroutine` 实现异步操作

In [6]:
import asyncio

async def hello():
    print("Hello world!")
    # 异步调用asyncio.sleep(1):
    await asyncio.sleep(1)
    print("Hello again!")

# asyncio.run(hello()) jupyter 内部已经有一个事件在 run 了，所以不能新创建事件了
# 方法一
await hello()

# 方法二
task = asyncio.create_task(hello())   # 交给已有的事件循环,后台运行
await task                            # 需要等结果时再 await

Hello world!
Hello again!
Hello world!
Hello again!


- 尝试改写之前的生产者消费者模型

In [7]:
import asyncio

async def consumer():
    r = ''
    while True:
        n = yield r                 # 和原来一样:产出 r,等待 asend 传入值
        if not n:
            return
        print('[CONSUMER] Consuming %s...' % n)
        await asyncio.sleep(0.1)    # 模拟耗时的 IO 操作
        r = '200 OK'

async def produce(c):
    await c.asend(None)             # 预激,对应原来的 c.send(None)
    n = 0
    while n < 5:
        n = n + 1
        print('[PRODUCER] Producing %s...' % n)
        r = await c.asend(n)        # 对应原来的 c.send(n)
        print('[PRODUCER] Consumer return: %s' % r)
    await c.aclose()                # 对应原来的 c.close()

async def main():
    c = consumer()
    await produce(c)

await main()

[PRODUCER] Producing 1...
[CONSUMER] Consuming 1...
[PRODUCER] Consumer return: 200 OK
[PRODUCER] Producing 2...
[CONSUMER] Consuming 2...
[PRODUCER] Consumer return: 200 OK
[PRODUCER] Producing 3...
[CONSUMER] Consuming 3...
[PRODUCER] Consumer return: 200 OK
[PRODUCER] Producing 4...
[CONSUMER] Consuming 4...
[PRODUCER] Consumer return: 200 OK
[PRODUCER] Producing 5...
[CONSUMER] Consuming 5...
[PRODUCER] Consumer return: 200 OK
